In [ ]:
!pip install nltk

In [ ]:
import pandas as pd
import sqlite3
import re
import openai
import time
import nltk
from difflib import SequenceMatcher
import csv
from openai import OpenAI

openai.api_key = ""

In [ ]:
fox = pd.read_csv('')
nyt = pd.read_csv('')
rebel = pd.read_csv('')
reuters = pd.read_csv('')
washpo = pd.read_csv('')

In [ ]:
outlets = ["fox", "nyt", "rebel", "reuters", "washpo"]
data_frames = {}
# Add your directory path here
directory_path = ''
for outlet in outlets:
    data_frames[outlet] = pd.read_csv(f'{directory_path}/{outlet}.csv', sep='|')
fox_data = data_frames['fox']
nyt_data = data_frames['nyt']
rebel_data = data_frames['rebel']
reuters_data = data_frames['reuters']
washpo_data = data_frames['washpo']

In [ ]:
client = OpenAI(api_key="")

In [ ]:
import difflib

def get_list_diff(old_list, new_list):
    diff = []
    seqm = difflib.SequenceMatcher(None, old_list, new_list)
    
    for tag, i1, i2, j1, j2 in seqm.get_opcodes():
        if tag == 'replace':
            # Handle replacement case: removed (-) and added (+)
            # Only mark as replaced if there's a mismatch in both segments
            if i2 - i1 == j2 - j1 and all(old_list[i] == new_list[j] for i, j in zip(range(i1, i2), range(j1, j2))):
                # In this case, it's actually an "equal"
                for i in range(i1, i2):
                    diff.append({'tag': '=', 'text': old_list[i]})
            else:
                for i in range(i1, i2):
                    diff.append({'tag': '-', 'text': old_list[i]})
                for j in range(j1, j2):
                    diff.append({'tag': '+', 'text': new_list[j]})
        elif tag == 'delete':
            # Words deleted: removed (-)
            for i in range(i1, i2):
                diff.append({'tag': '-', 'text': old_list[i]})
        elif tag == 'insert':
            # Words inserted: added (+)
            for j in range(j1, j2):
                diff.append({'tag': '+', 'text': new_list[j]})
        elif tag == 'equal':
            # Words unchanged: present (=)
            for i in range(i1, i2):
                diff.append({'tag': '=', 'text': old_list[i]})
    
    return diff


In [ ]:
def get_words(text, split_method='spacy'):
    words = text.split()
    cleaned_words = [re.sub(r'^\W+|\W+$', '', word) for word in words]
    return cleaned_words

def is_punctuation(word):
        punctuation_chars = ['!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', ':', ';', '<', '=', '>', '?', '@', '[', '\\', ']', '^', '_', '`', '{', '|', '}', '~']
        return all(char in punctuation_chars for char in word)

def get_word_diffs(s_old, s_new):
    s_old_words, s_new_words = get_words(s_old), get_words(s_new)
    return get_list_diff(s_old_words, s_new_words)

def clean_headline(headline):
    headline_cleaned = headline.split(" | ")[0].strip()
    return headline_cleaned

In [ ]:
def remove_common_words(added_words, removed_words):
    # Convert lists to sets to find common words
    added_set = set(added_words)
    removed_set = set(removed_words)
    
    # Find common words
    common_words = added_set & removed_set
    
    # Remove common words from both lists
    added_words = [word for word in added_words if word not in common_words]
    removed_words = [word for word in removed_words if word not in common_words]
    added_words_list = ', '.join(added_words)
    removed_words_list = ', '.join(removed_words)
    return added_words_list, removed_words_list

In [ ]:
def get_prompt(s_old, s_new):
    # Get word differences between the old and new strings
    word_diffs = get_word_diffs(s_old, s_new)
    
    # Initialize the word statistics output
    word_stat_output = {
        'num_removed_words': sum(1 for word in word_diffs if word['tag'] == '-'),
        'num_added_words': sum(1 for word in word_diffs if word['tag'] == '+'),
        'len_old_sent': len([word for word in word_diffs if word['tag'] != '+' and not is_punctuation(word['text'])]),
        'len_new_sent': len([word for word in word_diffs if word['tag'] != '-' and not is_punctuation(word['text'])]),
        's_old': [word for word in word_diffs if word['tag'] != '+' and not is_punctuation(word['text'])],
        's_new': [word for word in word_diffs if word['tag'] != '-' and not is_punctuation(word['text'])],
    }

    # Debug print to check the intermediate values
    print(word_stat_output)

    # Extract the added and removed words
    added_words = [word['text'] for word in word_diffs if word['tag'] == '+']
    removed_words = [word['text'] for word in word_diffs if word['tag'] == '-']

    
    # Convert the lists to strings
    added_words_list, removed_words_list = remove_common_words(added_words, removed_words)
    # Construct the final prompt
    a = "Given a pre-edited headline, a post-edited headline, list of added and removed words. Provide the Parts of Speech (POS) of the word lists given. Additionally, analyze the changes to determine if they introduce or remove any of the 13 types of media bias as outlined by AllSides.com. Provide a brief explanation for your analysis."
    b = "Original Headline: "
    c = "Edited Headline: "
    d = "Added words: "
    e = "Removed words: "
    prompt = b + s_old + "\n" + c + s_new + "\n" + d + added_words_list + "\n" + e + removed_words_list + "\n"  
    return prompt

In [ ]:
df = nyt_data
prompts = []
for i in range(len(df)):
    old_headline = clean_headline(df['prev_title'][i])
    new_headline = clean_headline(df['title'][i])
    prompt = get_prompt(old_headline, new_headline)
    prompts.append(prompt)
    print(prompt)
    print('='*90)

In [ ]:
df_prompts = pd.DataFrame(prompts, columns=['prompt'])
df_prompts.to_csv('')